In [ ]:
!pip install --upgrade pip setuptools wheel

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu130 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.


In [ ]:
!pip install -q datasets transformers seqeval accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/indic-ner"
CKPT_DIR = f"{DRIVE_DIR}/indicbert_hi_best"

tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR)
model = AutoModelForTokenClassification.from_pretrained(CKPT_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()

id2label = model.config.id2label
print("Device:", device)
print("Labels:", id2label)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Device: cuda
Labels: {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC'}


In [ ]:
import unicodedata
from datasets import load_dataset

LANGS = ["bn", "ta", "te"]

def load_split(lang: str, split: str):
    ds = load_dataset(
        "parquet",
        data_files={split: f"hf://datasets/ai4bharat/naamapadam@~parquet/{lang}/{split}/0000.parquet"},
    )[split]
    return ds

def normalize_tokens(example):
    example["tokens"] = [
        unicodedata.normalize("NFC", t) for t in example["tokens"]
    ]
    return example

raw = {}
for lang in LANGS:
    ds = load_split(lang, "validation")
    bad = sum(
        1 for ex in ds
        if len(ex["tokens"]) != len(ex["ner_tags"])
    )
    assert bad == 0, f"{lang}: {bad} misaligned rows"
    ds = ds.map(normalize_tokens, desc=f"NFC {lang}")
    raw[lang] = ds
    print(lang, len(ds), "rows | labels:", ds.features["ner_tags"].feature.names)

bn/validation/0000.parquet: reconstructing file:   0%|          |  0.00B /  470kB            

bn/validation/0000.parquet: downloading bytes:           |  0.00B            

Generating validation split: 0 examples [00:00, ? examples/s]

NFC bn:   0%|          | 0/4859 [00:00<?, ? examples/s]

bn 4859 rows | labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


...

te/validation/0000.parquet: reconstructing file:   0%|          |  0.00B /  220kB            

te/validation/0000.parquet: downloading bytes:           |  0.00B            

Generating validation split: 0 examples [00:00, ? examples/s]

NFC te:   0%|          | 0/2700 [00:00<?, ? examples/s]

te 2700 rows | labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [ ]:
MAX_LEN = 128

def tokenize_and_align(batch):
    enc = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LEN,
    )
    aligned = []
    for i, tags in enumerate(batch["ner_tags"]):
        word_ids = enc.word_ids(batch_index=i)
        labels, prev = [], None
        for wid in word_ids:
            if wid is None:
                labels.append(-100)
            elif wid != prev:
                labels.append(tags[wid])
            else:
                labels.append(-100)
            prev = wid
        aligned.append(labels)
    enc["labels"] = aligned
    return enc

tokenized = {
    lang: ds.map(
        tokenize_and_align,
        batched=True,
        remove_columns=ds.column_names,
        desc=f"Tokenizing {lang}",
    )
    for lang, ds in raw.items()
}

Tokenizing bn:   0%|          | 0/4859 [00:00<?, ? examples/s]

Tokenizing ta:   0%|          | 0/2795 [00:00<?, ? examples/s]

Tokenizing te:   0%|          | 0/2700 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForTokenClassification, Trainer

collator = DataCollatorForTokenClassification(tokenizer)
trainer = Trainer(model=model, data_collator=collator)

predictions = {}
for lang, ds in tokenized.items():
    out = trainer.predict(ds)
    preds = np.argmax(out.predictions, axis=-1)
    y_true, y_pred = [], []
    for pred, lab in zip(preds, out.label_ids):
        keep = lab != -100
        y_true.append([id2label[l] for l in lab[keep]])
        y_pred.append([id2label[p] for p in pred[keep]])
    predictions[lang] = (y_true, y_pred)
    print(lang, "done")

bn done


ta done


te done


In [ ]:
from seqeval.metrics import classification_report, f1_score

results = {}
for lang, (y_true, y_pred) in predictions.items():
    results[lang] = f1_score(y_true, y_pred)
    print(f"===== {lang} | zero-shot entity-F1: {results[lang]:.4f} =====")
    print(classification_report(y_true, y_pred))

===== bn | zero-shot entity-F1: 0.7361 =====
              precision    recall  f1-score   support

         LOC       0.72      0.78      0.75      2810
         ORG       0.59      0.62      0.61      1749
         PER       0.78      0.80      0.79      3696

   micro avg       0.72      0.76      0.74      8255
   macro avg       0.70      0.73      0.71      8255
weighted avg       0.72      0.76      0.74      8255

===== ta | zero-shot entity-F1: 0.7216 =====
              precision    recall  f1-score   support

         LOC       0.75      0.81      0.78      1539
         ORG       0.52      0.54      0.53      1024
         PER       0.75      0.81      0.78      1625

   micro avg       0.70      0.75      0.72      4188
   macro avg       0.68      0.72      0.70      4188
weighted avg       0.70      0.75      0.72      4188

===== te | zero-shot entity-F1: 0.7408 =====
              precision    recall  f1-score   support

         LOC       0.71      0.82      0.76     

In [ ]:
print("Zero-shot transfer (train: hi, IndicBERT)")
print("-" * 34)
print(f"{'target lang':<14}{'entity-F1':>10}")
print("-" * 34)
print(f"{'hi (in-language)':<14}{0.78:>10.4f}")
for lang, f1 in results.items():
    print(f"{lang:<14}{f1:>10.4f}")
print("-" * 34)

Zero-shot transfer (train: hi, IndicBERT)
----------------------------------
target lang    entity-F1
----------------------------------
hi (in-language)    0.7800
bn                0.7361
ta                0.7216
te                0.7408
----------------------------------


In [ ]:
import json

out = {
    "model": "indicbert_hi_best (IndicBERTv2-MLM-only, trained on hi 100k)",
    "eval_data": "validation splits (dev)",
    "in_language": {"hi": 0.78},
    "zero_shot": results,
}
path = f"{DRIVE_DIR}/zero_shot_results.json"
with open(path, "w") as f:
    json.dump(out, f, indent=2)
print("Saved:", path)

Saved: /content/drive/MyDrive/indic-ner/zero_shot_results.json
